In [1]:
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import Draw
from rdkit import DataStructs
from rdkit.Chem.Draw import SimilarityMaps
from rdkit.Chem.Draw import rdMolDraw2D
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator

In [ ]:
# 单一分子计算相似性

# 从mol文件读取分子对象
ABZ = Chem.MolFromMolFile('E:\DeepLearning\Exercise\Rdkit\Object\ABZ.mol')
H2 = Chem.MolFromMolFile('E:\DeepLearning\Exercise\Rdkit\Object\H2.mol')

# 将放入列表
ms = [ABZ, H2]
# 为每个分子计算RDKit指纹
fps = [Chem.RDKFingerprint(x) for x in ms]
# 计算两个指纹之间的Tanimoto相似度
DataStructs.FingerprintSimilarity(fps[0], fps[1])

# 相似度计算函数
def Obtainsimilarity():
    # 定义要使用的相似度度量列表
    metric_list = ['DataStructs.TanimotoSimilarity',
              'DataStructs.DiceSimilarity',
              'DataStructs.CosineSimilarity',
              'DataStructs.SokalSimilarity',
              'DataStructs.RusselSimilarity',
              'DataStructs.KulczynskiSimilarity',
              'DataStructs.McConnaugheySimilarity']
    # 遍历每个度量，计算并打印相似度
    for i in metric_list:
        print(f"{i}: {DataStructs.FingerprintSimilarity(fps[0], fps[1], metric=eval(i))}")

# 调用函数
Obtainsimilarity()

DataStructs.TanimotoSimilarity: 0.4262899262899263
DataStructs.DiceSimilarity: 0.5977605512489234
DataStructs.CosineSimilarity: 0.6067397946626676
DataStructs.SokalSimilarity: 0.27088212334113976
DataStructs.RusselSimilarity: 0.16943359375
DataStructs.KulczynskiSimilarity: 0.6158539195303901
DataStructs.McConnaugheySimilarity: 0.23170783906078024


In [5]:
# 批量计算分子相似性

from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import Draw
from rdkit import DataStructs
from rdkit.Chem.Draw import SimilarityMaps
from rdkit.Chem.Draw import rdMolDraw2D
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
import pathlib
import pandas as pd


# 批量读取mol文件函数
def load_mols_pathlib(mol_dir):
    # 获取文件夹下所有 .mol 文件
    mol_paths = pathlib.Path(mol_dir).glob("*.mol")
    # 新建空列表存放分子
    mols = []
    # 遍历每个mol文件并读取
    for p in mol_paths:
        mol = Chem.MolFromMolFile(str(p))
        # 读取成功加入列表
        if mol:
            mols.append(mol)
    # 返回所有分子
    return mols

# 获取文件名函数
def get_filenames(mol_dir):
    filenames = []
    mol_dir_path = pathlib.Path(mol_dir)
    for file_path in mol_dir_path.glob("*.mol"):
        filenames.append(file_path.stem)
    return filenames

# 计算相似性函数
def calculate_similarities(molecules, filenames):
    """
    molecules : list
        分子列表
    filenames : list
        文件名列表
    """
    # 定义相似性度量方法
    metric_list = [
        'DataStructs.TanimotoSimilarity',
        'DataStructs.DiceSimilarity',
        'DataStructs.CosineSimilarity',
        'DataStructs.SokalSimilarity',
        'DataStructs.RusselSimilarity',
        'DataStructs.KulczynskiSimilarity',
        'DataStructs.McConnaugheySimilarity'
    ]
    
    # 生成所有分子的指纹
    fpg = GetMorganGenerator(radius=2, fpSize=2048)
    fps = [fpg.GetFingerprint(mol) for mol in molecules]
    
    # 初始化结果列表
    results = []
    
    print("开始计算所有分子对之间的相似性...")
    print("-" * 50)
    
    # 只计算上三角矩阵，避免重复计算（如A-B和B-A）
    for i in range(len(molecules)):
        for j in range(i + 1, len(molecules)):  # j从i+1开始，避免重复
            # 获取文件名
            name_i = filenames[i] if filenames else f"分子_{i}"
            name_j = filenames[j] if filenames else f"分子_{j}"
            
            # 计算各种相似性度量
            row = {'分子1': name_i, '分子2': name_j, '分子对': f"{name_i}与{name_j}"}
            
            for metric_name in metric_list:
                try:
                    # 动态获取相似性函数
                    metric_func = eval(metric_name)
                    similarity = metric_func(fps[i], fps[j])
                    row[metric_name.split('.')[-1]] = similarity
                except Exception as e:
                    row[metric_name.split('.')[-1]] = None
                    print(f"  计算 {name_i}-{name_j} 的 {metric_name} 时出错: {e}")
            
            results.append(row)
            
    
    print(f"计算完成！共计算了 {len(results)} 个分子对")
    print("-" * 50)
    
    # 返回结果DataFrame
    return pd.DataFrame(results)


In [ ]:
# 主程序：
mol_dir = r"E:\DeepLearning\Exercise\Rdkit\Object"  # 请根据需要修改目录
molecules = load_mols_pathlib(mol_dir)
filenames = get_filenames(mol_dir)

print(f"成功读取了 {len(molecules)} 个分子")
print("文件名:", filenames)

if len(molecules) > 0:
    # 计算所有分子对之间的相似性
    results_df = calculate_similarities(molecules, filenames)

    # 保存到桌面（xlsx格式）
    import os
    desktop_path = os.path.join(os.path.expanduser('~'), 'Desktop')
    output_file = os.path.join(desktop_path, 'similarity_results.xlsx')

    try:
        results_df.to_excel(output_file, index=False)
        print(f"结果已保存到 {output_file}")
    except Exception as e:
        print(f"保存失败: {e}")
else:
    print("没有读取到任何分子")